In [0]:
# ============================================
# NOTEBOOK 3: Gold Layer - Analytics
# PROJECT: Wikipedia Clickstream Analytics
# ============================================

# Load silver tables
silver_clickstream = spark.table("workspace.default.silver_clickstream")
silver_pageviews = spark.table("workspace.default.silver_pageviews")

print("Silver Clickstream count:", silver_clickstream.count())
print("Silver Pageviews count:", silver_pageviews.count())
print("✅ Silver tables loaded successfully!")

Silver Clickstream count: 24522123
Silver Pageviews count: 861793
✅ Silver tables loaded successfully!


In [0]:
# ============================================
# GOLD: Query 1 - Top 10 Most Viewed Articles
# ============================================
from pyspark.sql.functions import col, sum, desc

gold_top_pages = silver_pageviews \
    .groupBy("article_title") \
    .agg(sum("view_count").alias("total_views")) \
    .orderBy(desc("total_views")) \
    .limit(10)

gold_top_pages.write.format("delta").mode("overwrite") \
    .saveAsTable("workspace.default.gold_top_pages")

print("✅ Query 1 Complete — Top 10 Most Viewed Articles:")
display(gold_top_pages)

✅ Query 1 Complete — Top 10 Most Viewed Articles:


article_title,total_views
Main_Page,57158
-,9927
Apple_Network_Server,2803
Deaths_in_2024,2545
Halloween,2404
Diwali,2373
Fascism,2269
2024_United_States_presidential_election,2244
Raindrop_cake,1904
Agatha_All_Along_(miniseries),1852


In [0]:
# ============================================
# GOLD: Query 2 - Top Entry Pages (via "other" type)
# ============================================
from pyspark.sql.functions import col, sum, desc

gold_entry_points = silver_clickstream \
    .filter(col("click_type") == "other") \
    .groupBy("target_page") \
    .agg(sum("click_count").alias("total_entries")) \
    .orderBy(desc("total_entries")) \
    .limit(10)

gold_entry_points.write.format("delta").mode("overwrite") \
    .saveAsTable("workspace.default.gold_entry_points")

print("✅ Query 2 Complete — Top Entry Pages:")
display(gold_entry_points)

✅ Query 2 Complete — Top Entry Pages:


target_page,total_entries
Main_Page,5214733
Hurricane_Milton,138558
2024_Israeli_invasion_of_Lebanon,127120
Yahya_Sinwar,126876
Killing_of_Yahya_Sinwar,107196
Joker:_Folie_à_Deux,106519
October_2024_Iranian_strikes_against_Israel,97385
Liam_Payne,88299
2024_Moldovan_European_Union_membership_referendum,82629
Shortness_of_breath,79802


In [0]:
# ============================================
# GOLD: Query 3 - Top Exit Pages
# ============================================
from pyspark.sql.functions import col, sum, desc

gold_exit_points = silver_clickstream \
    .filter(col("click_type") == "link") \
    .groupBy("source_page") \
    .agg(sum("click_count").alias("total_exits")) \
    .orderBy(desc("total_exits")) \
    .limit(10)

gold_exit_points.write.format("delta").mode("overwrite") \
    .saveAsTable("workspace.default.gold_exit_points")

print("✅ Query 3 Complete — Top Exit Pages:")
display(gold_exit_points)

✅ Query 3 Complete — Top Exit Pages:


source_page,total_exits
Liam_Payne,2986965
Ratan_Tata,2923340
One_Direction,2623072
Main_Page,2557576
Deaths_in_2024,2479758
Joker:_Folie_à_Deux,2251969
Lyle_and_Erik_Menendez,2077405
Sean_Combs,1944773
2024_United_States_presidential_election,1776171
Monsters:_The_Lyle_and_Erik_Menendez_Story,1611407


In [0]:
# ============================================
# GOLD: Query 4 - Two-Step Click Paths A→B→C
# ============================================
from pyspark.sql.functions import col, desc

hop1 = silver_clickstream.select(
    col("source_page").alias("page_a"),
    col("target_page").alias("page_b"),
    col("click_count").alias("count1")
)

hop2 = silver_clickstream.select(
    col("source_page").alias("page_b2"),
    col("target_page").alias("page_c"),
    col("click_count").alias("count2")
)

gold_click_paths = hop1.join(hop2, hop1.page_b == hop2.page_b2) \
    .select(
        col("page_a"),
        col("page_b"),
        col("page_c"),
        (col("count1") + col("count2")).alias("total_clicks")
    ) \
    .orderBy(desc("total_clicks")) \
    .limit(10)

gold_click_paths.write.format("delta").mode("overwrite") \
    .saveAsTable("workspace.default.gold_click_paths")

print("✅ Query 4 Complete — Top Two-Step Navigation Paths:")
display(gold_click_paths)

✅ Query 4 Complete — Top Two-Step Navigation Paths:


page_a,page_b,page_c,total_clicks
One_Direction,Liam_Payne,Cheryl_(singer),1807395
Liam_Payne,Cheryl_(singer),Ashley_Cole,1360744
Cheryl_(singer),Liam_Payne,Cheryl_(singer),1334585
Liam_Payne,Cheryl_(singer),Liam_Payne,1334585
Deaths_in_2024,Main_Page,Deaths_in_2024,1316333
Main_Page,Deaths_in_2024,Main_Page,1316333
Main_Page,Deaths_in_2024,Liam_Payne,1290164
Wikipedia,Main_Page,Deaths_in_2024,1280516
Main_Page,Deaths_in_2024,Teri_Garr,1269399
Wiki,Main_Page,Deaths_in_2024,1269115


In [0]:
# ============================================
# GOLD: Query 5 & 6 - Traffic & Click Types
# ============================================
from pyspark.sql.functions import col, sum, desc, count

# Query 5 - Top pages by total views
gold_hourly_traffic = silver_pageviews \
    .groupBy("article_title") \
    .agg(sum("view_count").alias("total_views")) \
    .orderBy(desc("total_views")) \
    .limit(20)

gold_hourly_traffic.write.format("delta").mode("overwrite") \
    .saveAsTable("workspace.default.gold_hourly_traffic")

print("✅ Query 5 Complete — Top Pages Traffic:")
display(gold_hourly_traffic)

# Query 6 - Click type breakdown
gold_click_types = silver_clickstream \
    .groupBy("click_type") \
    .agg(
        sum("click_count").alias("total_clicks"),
        count("source_page").alias("num_connections")
    ) \
    .orderBy(desc("total_clicks"))

gold_click_types.write.format("delta").mode("overwrite") \
    .saveAsTable("workspace.default.gold_click_types")

print("✅ Query 6 Complete — Click Type Breakdown:")
display(gold_click_types)

print("🎉 Gold Layer Complete! All 6 queries done!")

✅ Query 5 Complete — Top Pages Traffic:


article_title,total_views
Main_Page,57158
-,9927
Apple_Network_Server,2803
Deaths_in_2024,2545
Halloween,2404
Diwali,2373
Fascism,2269
2024_United_States_presidential_election,2244
Raindrop_cake,1904
Agatha_All_Along_(miniseries),1852


✅ Query 6 Complete — Click Type Breakdown:


click_type,total_clicks,num_connections
link,2318940193,23663816
other,52561476,858307


🎉 Gold Layer Complete! All 6 queries done!
